In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

RESULTS_DIR = "/Users/yuhaopro/Projects/scraper-nextjs/agents/eval_results"
PLOTS_DIR   = "/Users/yuhaopro/Projects/scraper-nextjs/agents/eval_results/plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

def load(filename):
    with open(f"{RESULTS_DIR}/{filename}") as f:
        return json.load(f)

def save(fig, name):
    path = os.path.join(PLOTS_DIR, name)
    fig.savefig(path, format="eps", dpi=300, bbox_inches="tight")
    print(f"Saved → {path}")

In [ ]:
# ── Chart 1: Accuracy over pipeline versions ─────────────────────────────────

VERSION_FILES = [
    ("v1", "eval_20260315_145643_pipeline.json"),
    ("v2", "eval_20260315_152556_pipeline.json"),
    ("v3", "eval_20260315_155729_pipeline.json"),
    ("v4", "eval_20260315_180044_pipeline.json"),
]

versions   = [v for v, _ in VERSION_FILES]
accuracies = []
costs_m    = []

for _, fname in VERSION_FILES:
    d = load(fname)
    accuracies.append(d["accuracy"] * 100)
    costs_m.append(d["avg_cost"] * 1000)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(versions, accuracies, marker="o", color="steelblue", linewidth=2.5, markersize=8)
for v, a in zip(versions, accuracies):
    ax.annotate(f"{a:.0f}%", (v, a), textcoords="offset points", xytext=(0, 10), ha="center", fontsize=10)
ax.set_title("Accuracy by Version", fontsize=13, fontweight="bold")
ax.set_xlabel("Version")
ax.set_ylabel("Accuracy (%)")
ax.set_ylim(0, 100)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
save(fig, "accuracy_by_version.eps")
plt.show()

In [ ]:
# ── Chart 2: Avg Cost per Claim over pipeline versions ───────────────────────

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(versions, costs_m, marker="o", color="darkorange", linewidth=2.5, markersize=8)
for v, c in zip(versions, costs_m):
    ax.annotate(f"{c:.3f}", (v, c), textcoords="offset points", xytext=(0, 10), ha="center", fontsize=10)
ax.set_title("Avg Cost per Claim by Version", fontsize=13, fontweight="bold")
ax.set_xlabel("Version")
ax.set_ylabel("Cost (m$)")
ymin, ymax = min(costs_m), max(costs_m)
margin = (ymax - ymin) * 0.35
ax.set_ylim(ymin - margin * 0.5, ymax + margin)
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
save(fig, "cost_by_version.eps")
plt.show()

In [ ]:
# ── Charts 3a/3b/3c: Accuracy / Cost / Latency vs claim token length ─────────
# Baseline = basic eval (224858), Refined = pipeline eval (180044)

basic    = load("eval_20260314_224858_basic.json")
pipeline = load("eval_20260315_180044_pipeline.json")

def build_df(data, label):
    rows = []
    for r in data["records"]:
        text      = r.get("claim_text", "")
        token_len = len(text.split())
        rows.append({
            "token_len": token_len,
            "correct":   int(r.get("correct", False)),
            "cost_m":    sum(r.get("costs", {}).values()) * 1000,
            "latency":   r.get("latency_seconds", 0),
            "label":     label,
        })
    return pd.DataFrame(rows)

df_basic    = build_df(basic,    "Baseline (basic)")
df_pipeline = build_df(pipeline, "Refined pipeline")
df_all      = pd.concat([df_basic, df_pipeline], ignore_index=True)

bins       = np.linspace(df_all["token_len"].min(), df_all["token_len"].max(), 11)
bin_labels = [f"{int(bins[i])}–{int(bins[i+1])}" for i in range(len(bins) - 1)]
cat_type   = pd.CategoricalDtype(categories=bin_labels, ordered=True)

for df in [df_basic, df_pipeline]:
    df["bin"] = pd.cut(df["token_len"], bins=bins, labels=bin_labels, include_lowest=True)
    df["bin"] = df["bin"].astype(cat_type)

def agg(df):
    return (
        df.groupby("bin", observed=False)
        .agg(accuracy=("correct", "mean"), cost_m=("cost_m", "mean"),
             latency=("latency", "mean"), count=("correct", "count"))
        .reindex(bin_labels)
        .reset_index()
    )

agg_basic    = agg(df_basic)
agg_pipeline = agg(df_pipeline)

x     = np.arange(len(bin_labels))
width = 0.35

def token_chart(metric, ylabel, title, tick_fmt, val_fmt, filename,
                color_b="#4C9BE8", color_p="#E8834C"):
    fig, ax = plt.subplots(figsize=(13, 5))
    b_vals  = agg_basic[metric].fillna(0).values
    p_vals  = agg_pipeline[metric].fillna(0).values
    count_b = agg_basic["count"].fillna(0).values
    count_p = agg_pipeline["count"].fillna(0).values
    bars_b = ax.bar(x - width/2, b_vals, width, label="Baseline (basic)", color=color_b, alpha=0.85)
    bars_p = ax.bar(x + width/2, p_vals, width, label="Refined pipeline", color=color_p, alpha=0.85)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Claim token length (words)")
    ax.set_ylabel(ylabel)
    ax.set_xticks(x)
    ax.set_xticklabels(bin_labels, rotation=35, ha="right", fontsize=8)
    ax.yaxis.set_major_formatter(tick_fmt)
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.35)
    ymax = ax.get_ylim()[1]
    for bar, val, n in zip(bars_b, b_vals, count_b):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + ymax * 0.01,
                    f"{val_fmt(val)}\n(n={int(n)})", ha="center", va="bottom", fontsize=6.5, color="#2a6090")
    for bar, val, n in zip(bars_p, p_vals, count_p):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + ymax * 0.01,
                    f"{val_fmt(val)}\n(n={int(n)})", ha="center", va="bottom", fontsize=6.5, color="#903a2a")
    plt.tight_layout()
    save(fig, filename)
    plt.show()

token_chart(
    "accuracy", "Accuracy", "Accuracy by Claim Token Length",
    mticker.FuncFormatter(lambda v, _: f"{v*100:.0f}%"),
    lambda v: f"{v*100:.0f}%",
    "accuracy_by_token.eps",
)
token_chart(
    "cost_m", "Avg Cost (m$)", "Avg Cost by Claim Token Length",
    mticker.FormatStrFormatter("%.3f"),
    lambda v: f"{v:.3f}",
    "avg_cost_by_token.eps",
)
token_chart(
    "latency", "Avg Latency (s)", "Avg Latency by Claim Token Length",
    mticker.FormatStrFormatter("%.0f"),
    lambda v: f"{v:.0f}",
    "avg_latency_by_token.eps",
)